# Underfitting e overfitting



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '24c516651bf205b1c9511923ad4971fdca39a6248568d2b3aa46c198e64d1821'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=False)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


In questa lezione imparerai a riconoscere **underfitting** e **overfitting** e a usare questi concetti per migliorare le previsioni.

# Confrontare modelli diversi
Ora che sai misurare la qualità, puoi sperimentare. La [documentazione di DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html) elenca molti parametri. Fra i più importanti ci sono quelli che controllano la profondità, cioè il numero di divisioni lungo un percorso verso una foglia.

![Albero poco profondo](https://storage.googleapis.com/kaggle-media/learn/images/R3ywQsR.png)

Un albero può arrivare a dieci divisioni tra radice e foglia. Ogni livello può raddoppiare i gruppi: 2, 4, 8, fino a \(2^{10}=1024\) foglie. Più foglie significa meno case in ciascuna foglia.

Con pochissime case per foglia, le previsioni possono essere molto vicine ai prezzi del training, ma poco affidabili sui dati nuovi. Questo è **overfitting**: il modello si adatta quasi perfettamente ai dati di addestramento, comprese particolarità che non si ripetono.

Un albero troppo semplice, invece, distingue solo pochi gruppi molto eterogenei. Se divide tutte le case in due o quattro gruppi, può sbagliare anche sui dati di training. Questo è **underfitting**: non riesce a cogliere le relazioni importanti.

Cerchiamo quindi la complessità che minimizza l'errore di validazione, il minimo della curva rossa:

![Underfitting e overfitting](https://storage.googleapis.com/kaggle-media/learn/images/AXSEOfI.png)

# Esempio
Ci sono vari modi di controllare la profondità, anche consentendo percorsi diversi. **`max_leaf_nodes`** limita il numero di foglie. Aumentandolo ci spostiamo da modelli troppo semplici verso modelli che possono sovradattarsi. Usiamo una funzione per confrontare i MAE:


In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    mae = mean_absolute_error(val_y, preds_val)
    return(mae)

Carichiamo i dati nelle variabili **`train_X`**, **`val_X`**, **`train_y`** e **`val_y`** con il procedimento già visto.


In [ ]:
# Qui viene eseguito il caricamento dei dati.
import pandas as pd
    
# Carica i dati.
melbourne_file_path = '../input/melbourne-housing-snapshot/melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path) 
# Escludi le righe con valori mancanti.
filtered_melbourne_data = melbourne_data.dropna(axis=0)
# Scegli il target e le feature.
y = filtered_melbourne_data.Price
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 
                        'YearBuilt', 'Lattitude', 'Longtitude']
X = filtered_melbourne_data[melbourne_features]

from sklearn.model_selection import train_test_split

# Dividi feature e target nei set di training e validazione.
train_X, val_X, train_y, val_y = train_test_split(X, y,random_state = 0)

Un ciclo `for` confronta i modelli con diversi valori di **`max_leaf_nodes`**.


In [ ]:
# Confronta il MAE per diversi valori di max_leaf_nodes.
for max_leaf_nodes in [5, 50, 500, 5000]:
    my_mae = get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    print("Max leaf nodes: %d  \t\t Mean Absolute Error:  %d" %(max_leaf_nodes, my_mae))

Fra i valori provati nell'esempio, il migliore è 500 foglie.

# Riepilogo
Un modello può soffrire di:

* **Overfitting:** apprende associazioni occasionali che non si ripetono nei dati futuri.
* **Underfitting:** non riesce a rappresentare relazioni rilevanti.

La validazione usa dati esclusi dal training per confrontare più modelli e scegliere quello migliore.

Ora prova a **[ottimizzare il modello costruito in precedenza](https://prof-davidaffo.github.io/kaggle-learntools/corsi/machine_learning/ex5.html)**.
